In [ ]:
# run this to see all the outputs below each cell, not only the last instruction 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [43]:
#_____________________________________ part a_1 _________________________________________#
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torchvision import transforms
import numpy as np
from PIL import Image

print("GPU available: {}".format(torch.cuda.is_available()))
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
#device = 'cpu'

# Path to the folder containing the images
images_dir = './ByteToImage/'
if not(os.path.exists(images_dir)):
    os.system('unrar x ByteToImage.rar')
    print("ByteToImage.rar has been extracted")


# Load labels into a pandas DataFrame
labels_df = pd.read_csv('trainLabels.csv')

# Get list of image paths and labels
X = [os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.png')]
y = []

if not(os.path.isfile("new_labels.csv")):
    print("arranging the labels")
    for i in range(len(X)):
        img_id=str(X[i].split('/')[-1].split('.')[0])
        for j in range(len(labels_df['Class'])):
            if img_id==labels_df['Id'][j]:
                y.append(labels_df['Class'][j])
    np.savetxt("new_labels.csv", y, delimiter=",")
    print("new_labels.csv has been made")

data = np.loadtxt("new_labels.csv", delimiter=",")
y = [int(x) for x in data.tolist()]

# Split data into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=15)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=15)

# Define a custom function to load and transform data
def load_and_transform_data(X, y, transform):
    data = []
    for i in range(len(X)):
        img_path = X[i]
        img = Image.open(img_path).convert('RGB')
        if transform is not None:
            img = transform(img)
        label = y[i]-1 # labels are in [1,9], should be in [0,8]
        data.append((img, label))
    return data

# Load images and apply transformations # also you can use transforms.Resize((64, 64))
data_transforms = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                                      transforms.Resize((64, 64)),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean=(0.5, ), std=(0.5, ))])

train_data = load_and_transform_data(X_train, y_train, transform=data_transforms)
val_data = load_and_transform_data(X_val, y_val, transform=data_transforms)
test_data = load_and_transform_data(X_test, y_test, transform=data_transforms)

batch_size=128

trainloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
valloader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

print("Training set size:", len(train_data))
print("Validation set size:", len(val_data))
print("Test set size:", len(test_data))
print(type(train_data), type(train_data[0]), type(train_data[0][0]))
print("image shape:", train_data[0][0].shape)
print("label type:", type(train_data[0][1]))
print(train_data[0][0].shape)
#print(len(list(trainloader)))


GPU available: True
Training set size: 8688
Validation set size: 1086
Test set size: 1086
<class 'list'> <class 'tuple'> <class 'torch.Tensor'>
image shape: torch.Size([1, 64, 64])
label type: <class 'int'>
torch.Size([1, 64, 64])


In [5]:
#_____________________________________ part a_2 _________________________________________#
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class ParametricCNN(nn.Module):
    def __init__(self, num_classes=9, lr=0.001, reg=0, device='cpu'):
        super(ParametricCNN, self).__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)
        self.pool6 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv7 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn7 = nn.BatchNorm2d(512)
        self.conv8 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn8 = nn.BatchNorm2d(512)
        self.pool8 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layer
        self.fc1 = nn.Linear(512 * 4 * 4, 320)
        self.fcbn1 = nn.BatchNorm1d(320)
        self.fc2 = nn.Linear(320, 320)
        self.fcbn2 = nn.BatchNorm1d(320)
        self.fc3 = nn.Linear(320, 320)
        self.fcbn3 = nn.BatchNorm1d(320)
        self.fc4 = nn.Linear(320, num_classes)

        # Initialize the weights and biases using Xavier initialization
        nn.init.xavier_uniform_(self.conv1.weight)
        nn.init.xavier_uniform_(self.conv2.weight)
        nn.init.xavier_uniform_(self.conv3.weight)
        nn.init.xavier_uniform_(self.conv4.weight)
        nn.init.xavier_uniform_(self.conv5.weight)
        nn.init.xavier_uniform_(self.conv6.weight)
        nn.init.xavier_uniform_(self.conv7.weight)
        nn.init.xavier_uniform_(self.conv8.weight)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)
        nn.init.xavier_uniform_(self.fc4.weight)
        nn.init.zeros_(self.conv1.bias)
        nn.init.zeros_(self.conv2.bias)
        nn.init.zeros_(self.conv3.bias)
        nn.init.zeros_(self.conv4.bias)
        nn.init.zeros_(self.conv5.bias)
        nn.init.zeros_(self.conv6.bias)
        nn.init.zeros_(self.conv7.bias)
        nn.init.zeros_(self.conv8.bias)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)
        nn.init.zeros_(self.fc4.bias)

        # Hyperparameters
        self.lr = lr
        self.reg = reg
        self.num_classes = num_classes
        self.device = device

    def forward(self, x):
        x = self.pool2(F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))))
        x = self.pool4(F.relu(self.bn4(self.conv4(F.relu(self.bn3(self.conv3(x)))))))
        x = self.pool6(F.relu(self.bn6(self.conv6(F.relu(self.bn5(self.conv5(x)))))))
        x = self.pool8(F.relu(self.bn8(self.conv8(F.relu(self.bn7(self.conv7(x)))))))
        x = x.view(-1, 512 * 4 * 4)
        x = F.relu(self.fcbn1(self.fc1(x)))
        x = F.relu(self.fcbn2(self.fc2(x)))
        x = F.relu(self.fcbn3(self.fc3(x)))
        x = F.softmax(self.fc4(x), dim=1)
        return x
    
    def configure_optimizers(self):
        optimizer = optim.SGD(self.parameters(), lr=self.lr, weight_decay=self.reg)
        return optimizer


In [44]:
#_____________________________________ part a_2 _________________________________________#
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class ParametricCNN(nn.Module):
    def __init__(self, num_classes=9, lr=0.001, reg=0, device='cpu'):
        super(ParametricCNN, self).__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layer
        self.fc1 = nn.Linear(32 * 32 * 32, 512)
        self.fcbn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(p=0.5) # dropout probability of 0.5
        self.fc2 = nn.Linear(512, num_classes)

        # Initialize the weights and biases using Xavier initialization
        nn.init.xavier_uniform_(self.conv1.weight)
        nn.init.xavier_uniform_(self.conv2.weight)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.conv1.bias)
        nn.init.zeros_(self.conv2.bias)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)

        # Hyperparameters
        self.lr = lr
        self.reg = reg
        self.num_classes = num_classes
        self.device = device

    def forward(self, x):
        x = self.pool2(F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))))
        x = x.view(-1, 32 * 32 * 32)
        x = F.relu(self.fcbn1(self.fc1(x)))
        x = self.dropout1(x)
        x = F.softmax(self.fc2(x), dim=1)
        return x
    
    def configure_optimizers(self):
        optimizer = optim.SGD(self.parameters(), lr=self.lr, weight_decay=self.reg)
        return optimizer
    

In [45]:
#_____________________________________ part b_1 _________________________________________#

net = ParametricCNN(lr=0.01, reg=0.001, device=device)
net.to(device)
print(f'num of parameters: {sum(p.numel() for p in net.parameters())}')
optimizer = net.configure_optimizers()

criterion = nn.CrossEntropyLoss()

epochs = 20

epoch_log = []
loss_log = []
train_accuracy_log = []
test_accuracy_log = []
validation_accuracy_log = []

for epoch in range(epochs):  
    print(f'Starting Epoch: {epoch+1}...')

    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data

        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()

        outputs = net(inputs) # Forward Propagation 
        loss = criterion(outputs, labels) # Get Loss (quantify the difference between the results and predictions)
        loss.backward() # Back propagate to obtain the new gradients for all nodes 
        optimizer.step() # Update the gradients/weights
        if i==0: 
            for name, param in net.named_parameters():
                print(name, param.grad.norm())
        running_loss += loss.item()
        
    correct = 0 
    total = 0 

    with torch.no_grad():
        for data in trainloader:
            images, labels = data

            images = images.to(device)
            labels = labels.to(device)

            outputs = net(images)

            _, predicted = torch.max(outputs.data, dim = 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        epoch_num = epoch + 1
        NumOfBatches=len(train_data)/batch_size
        actual_loss = running_loss / NumOfBatches

        print(f'Loss: {actual_loss:.3f}'+ '\n'+ f'Train Accuracy = {train_accuracy:.3f}%')
        running_loss = 0.0

    correct = 0 
    total = 0 

    with torch.no_grad():
        for data in valloader:
            images, labels = data

            images = images.to(device)
            labels = labels.to(device)

            outputs = net(images)

            _, predicted = torch.max(outputs.data, dim = 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        validation_accuracy = 100 * correct / total

        print(f'Validation Accuracy = {validation_accuracy:.3f}%')

    correct = 0 
    total = 0 

    with torch.no_grad():
        for data in testloader:
            images, labels = data

            images = images.to(device)
            labels = labels.to(device)

            outputs = net(images)

            _, predicted = torch.max(outputs.data, dim = 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total

        print(f'Test Accuracy = {test_accuracy:.3f}%')

    epoch_log.append(epoch_num)
    loss_log.append(actual_loss)
    train_accuracy_log.append(train_accuracy)
    validation_accuracy_log.append(validation_accuracy)
    test_accuracy_log.append(test_accuracy)

print('Finished Training')


ParametricCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32768, out_features=512, bias=True)
  (fcbn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=9, bias=True)
)

num of parameters: 16788265
Starting Epoch: 1...
conv1.weight tensor(0.5961, device='cuda:0')
conv1.bias tensor(2.9225e-06, device='cuda:0')
bn1.weight tensor(0.0321, device='cuda:0')
bn1.bias tensor(0.0313, device='cuda:0')
conv2.weight tensor(0.5047, device='cuda:0')
conv2.bias tensor(9.7000e-08, device='cuda:0')
bn2.weight tensor(0.0364, device='cuda:0')
bn2.bias tensor(0.0208, device='cuda:0')
fc1.weight tensor(3.9383, device='cuda:0')
fc1.bias tensor(2.3949e-09, device='cuda:0')
fcbn1.weight tensor(0.0296, device='cuda:0')
fcbn1.bias tensor(0.0324, device='cuda:0')
fc2.weight tensor(0.4998, device='cuda:0')
fc2.bias tensor(0.0212, device='cuda:0')
Loss: 1.675
Train Accuracy = 82.528%
Validation Accuracy = 80.939%
Test Accuracy = 81.400%
Starting Epoch: 2...
conv1.weight tensor(0.2664, device='cuda:0')
conv1.bias tensor(1.1029e-06, device='cuda:0')
bn1.weight tensor(0.0126, device='cuda:0')
bn1.bias tensor(0.0116, device='cuda:0')
conv2.weight tensor(0.2350, device='cuda:0')
conv2.

In [34]:
#_____________________________________ part g _________________________________________#
import torchvision.models as models
import torch.optim as optim
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torchvision import transforms
import numpy as np
from PIL import Image

print("GPU available: {}".format(torch.cuda.is_available()))
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
device = 'cpu'

# Path to the folder containing the images
images_dir = './ByteToImage/'
if not(os.path.exists(images_dir)):
    os.system('unrar x ByteToImage.rar')
    print("ByteToImage.rar has been extracted")


# Load labels into a pandas DataFrame
labels_df = pd.read_csv('trainLabels.csv')

# Get list of image paths and labels
X = [os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.png')]
y = []

if not(os.path.isfile("new_labels.csv")):
    print("arranging the labels")
    for i in range(len(X)):
        img_id=str(X[i].split('/')[-1].split('.')[0])
        for j in range(len(labels_df['Class'])):
            if img_id==labels_df['Id'][j]:
                y.append(labels_df['Class'][j])
    np.savetxt("new_labels.csv", y, delimiter=",")
    print("new_labels.csv has been made")

data = np.loadtxt("new_labels.csv", delimiter=",")
y = [int(x) for x in data.tolist()]

# Split data into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=15)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=15)

# Define a custom function to load and transform data
def load_and_transform_data(X, y, transform):
    data = []
    for i in range(len(X)):
        img_path = X[i]
        img = Image.open(img_path).convert('RGB')
        if transform is not None:
            img = transform(img)
        label = y[i]-1 # labels are in [1,9], should be in [0,8]
        data.append((img, label))
    return data

# Load images and apply transformations # also you can use transforms.Resize((64, 64))
data_transforms = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize(mean=(0.5, ), std=(0.5, ))])

train_data = load_and_transform_data(X_train, y_train, transform=data_transforms)
val_data = load_and_transform_data(X_val, y_val, transform=data_transforms)
test_data = load_and_transform_data(X_test, y_test, transform=data_transforms)

batch_size=128

trainloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
valloader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

print("Training set size:", len(train_data))
print("Validation set size:", len(val_data))
print("Test set size:", len(test_data))
print(type(train_data), type(train_data[0]), type(train_data[0][0]))
print("image shape:", train_data[0][0].shape)
print("label type:", type(train_data[0][1]))
print(train_data[0][0].shape)
#print(len(list(trainloader)))

# Load pre-trained MobileNet-v2 model
model = models.mobilenet_v2(pretrained=True)

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 3 layers for training
for param in model.features[-3:].parameters():
    param.requires_grad = True

# Replace last layer for 9-class classification
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, 9)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

criterion = nn.CrossEntropyLoss()

epochs = 100

epoch_log = []
loss_log = []
train_accuracy_log = []
test_accuracy_log = []
validation_accuracy_log = []

for epoch in range(epochs):  
    print(f'Starting Epoch: {epoch+1}...')

    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data

        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()

        outputs = model(inputs) # Forward Propagation 
        loss = criterion(outputs, labels) # Get Loss (quantify the difference between the results and predictions)
        loss.backward() # Back propagate to obtain the new gradients for all nodes 
        optimizer.step() # Update the gradients/weights
        running_loss += loss.item()
        
    correct = 0 
    total = 0 

    with torch.no_grad():
        for data in trainloader:
            images, labels = data

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, dim = 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        epoch_num = epoch + 1
        NumOfBatches=len(train_data)/batch_size
        actual_loss = running_loss / NumOfBatches

        print(f'Loss: {actual_loss:.3f}'+ '\n'+ f'Train Accuracy = {train_accuracy:.3f}%')
        running_loss = 0.0

    correct = 0 
    total = 0 

    with torch.no_grad():
        for data in valloader:
            images, labels = data

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, dim = 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        validation_accuracy = 100 * correct / total

        print(f'Validation Accuracy = {validation_accuracy:.3f}%')

    correct = 0 
    total = 0 

    with torch.no_grad():
        for data in testloader:
            images, labels = data

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, dim = 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total

        print(f'Test Accuracy = {test_accuracy:.3f}%')

    epoch_log.append(epoch_num)
    loss_log.append(actual_loss)
    train_accuracy_log.append(train_accuracy)
    validation_accuracy_log.append(validation_accuracy)
    test_accuracy_log.append(test_accuracy)

print('Finished Training')


GPU available: True
Training set size: 8688
Validation set size: 1086
Test set size: 1086
<class 'list'> <class 'tuple'> <class 'torch.Tensor'>
image shape: torch.Size([3, 32, 32])
label type: <class 'int'>
torch.Size([3, 32, 32])
Starting Epoch: 1...


/home/nima/anaconda3/envs/torchenv3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/nima/anaconda3/envs/torchenv3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loss: 0.512
Train Accuracy = 90.527%
Validation Accuracy = 89.134%
Test Accuracy = 88.306%
Starting Epoch: 2...
Loss: 0.288
Train Accuracy = 92.208%
Validation Accuracy = 90.331%
Test Accuracy = 90.055%
Starting Epoch: 3...
Loss: 0.235
Train Accuracy = 94.326%
Validation Accuracy = 90.331%
Test Accuracy = 89.871%
Starting Epoch: 4...
Loss: 0.195
Train Accuracy = 94.959%
Validation Accuracy = 90.792%
Test Accuracy = 90.700%
Starting Epoch: 5...
Loss: 0.172
Train Accuracy = 96.064%
Validation Accuracy = 92.449%
Test Accuracy = 91.068%
Starting Epoch: 6...
Loss: 0.141
Train Accuracy = 96.512%
Validation Accuracy = 93.186%
Test Accuracy = 90.516%
Starting Epoch: 7...
Loss: 0.134
Train Accuracy = 96.282%
Validation Accuracy = 91.713%
Test Accuracy = 91.713%
Starting Epoch: 8...
Loss: 0.117
Train Accuracy = 96.743%
Validation Accuracy = 92.265%
Test Accuracy = 90.792%
Starting Epoch: 9...
Loss: 0.114
Train Accuracy = 97.295%
Validation Accuracy = 92.173%
Test Accuracy = 90.700%
Starting Epoc